# Part 2: Multi-Source RAG with Query Routing

Intelligently routes queries to CSV (sales data) or text (product pages) based on question type.

**Architecture:** User Query → Query Router → CSV / Text / Both → LLM Answer

All pipeline logic lives in `src/part2_pipeline.py`.

## Step 1: Setup

In [4]:
import sys
import os

# Make sure the project root is on the path so src/ can be imported
sys.path.insert(0, os.path.abspath(".."))

from src.part2_pipeline import answer_question, classify_query
from src.config import CSV_PATH, TEXT_DIR, GROQ_MODEL

print("Imports OK")
print(f"  CSV  : {CSV_PATH}")
print(f"  Text : {TEXT_DIR}")
print(f"  Model: {GROQ_MODEL}")


Imports OK
  CSV  : ./data/structured/daily_sales.csv
  Text : ./data/unstructured/
  Model: llama-3.3-70b-versatile


## Step 2: Explore the Data

In [5]:
import glob
import pandas as pd

# Preview CSV structure
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Categories:", sorted(df["category"].unique()))
print("Regions:", sorted(df["region"].unique()))
df.head(3)


Shape: (1000, 8)
Columns: ['date', 'product_id', 'product_name', 'category', 'units_sold', 'unit_price', 'total_revenue', 'region']
Date range: 2024-10-03 to 2024-12-31
Categories: ['Beauty & Personal Care', 'Books', 'Clothing', 'Electronics', 'Food & Grocery', 'Home & Kitchen', 'Office Supplies', 'Pet Supplies', 'Sports & Outdoors', 'Toys & Games']
Regions: ['Central', 'East', 'North', 'South', 'West']


,date,product_id,product_name,category,units_sold,unit_price,total_revenue,region
0,2024-10-03,BOOK003,Mystery Novel Collection,Books,42,24.99,1049.58,Central
1,2024-10-03,ELEC002,USB-C Fast Charger,Electronics,57,24.99,1424.43,Central
2,2024-10-03,BOOK001,Python Programming Guide,Books,39,39.65,1546.35,West


In [6]:
# List available product text files
txt_files = sorted(glob.glob(os.path.join(TEXT_DIR, "*.txt")))
print(str(len(txt_files)) + " product page files found:")
for f in txt_files:
    print("  " + os.path.basename(f))

# Preview one file
with open(txt_files[0]) as f:
    print("\nPreview of " + os.path.basename(txt_files[0]))
    print("=" * 50)
    print(f.read()[:800])

10 product page files found:
  BEAU001_product_page.txt
  BOOK001_product_page.txt
  CLTH001_product_page.txt
  ELEC001_product_page.txt
  FOOD001_product_page.txt
  HOME003_product_page.txt
  OFFC001_product_page.txt
  PETS001_product_page.txt
  SPRT001_product_page.txt
  TOYS001_product_page.txt

Preview of BEAU001_product_page.txt
VITAMIN C SERUM - PRODUCT PAGE

Product: Vitamin C Serum
Brand: GlowLab Skincare
Price: $28.99
SKU: BEAU001
Category: Beauty & Personal Care

PRODUCT DESCRIPTION:
Reveal brighter, more youthful skin with GlowLab's Vitamin C Serum. This powerful
antioxidant formula combines 20% Vitamin C (L-Ascorbic Acid) with Vitamin E and
Hyaluronic Acid to reduce dark spots, fine lines, and sun damage while hydrating
and protecting your skin.

Key Ingredients:
- 20% L-Ascorbic Acid (Vitamin C) - brightens and evens skin tone
- Vitamin E - enhances antioxidant protection
- Hyaluronic Acid - deep hydration and plumping
- Ferulic Acid - stabilizes vitamins C and E, boosts e

## Step 3: Verify the Router

Confirm the router correctly classifies all 6 test questions before running them.

In [7]:
all_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

print("Router classification preview")
print("=" * 60)
for i, q in enumerate(all_questions, 1):
    result = classify_query(q)
    print(f"Q{i} [{result['route'].upper():4s}] {q[:55]}")
    print(f"     → {result['reasoning']}")
    print()


Router classification preview
Q1 [CSV ] What was the total revenue for Electronics category in 
     → The question is about total revenue, which is a numerical value related to sales, so it should be routed to the csv data source.

Q2 [CSV ] Which region had the highest sales volume?
     → The question is about sales volume, which is a numerical metric that can be found in the structured CSV data.

Q3 [TEXT] What are the key features of the Wireless Bluetooth Hea
     → The question is about product features, which is typically found in the product descriptions or specifications in the text data source.

Q4 [TEXT] What do customers say about the Air Fryer's ease of cle
     → The question is about customer reviews, which is purely about unstructured text data.

Q5 [BOTH] Which product has the best customer reviews and how wel
     → The question requires combining sales performance from the CSV data with customer review information from the text data to determine the product with the

## Step 4: Questions 1–2 (CSV Only)

Analytical questions answered entirely from the sales CSV.

In [8]:
csv_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
]

RESULTS_FILE = "./part2_results.txt"

# Start results file fresh
with open(RESULTS_FILE, "w") as f:
    f.write("Part 2: Multi-Source RAG Results\n" + "=" * 60 + "\n\n")

for i, q in enumerate(csv_questions, 1):
    answer = answer_question(q)
    print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
    print(answer)
    with open(RESULTS_FILE, "a") as f:
        f.write("Q" + str(i) + ": " + q + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")


Question: What was the total revenue for Electronics category in December 2024?

[Router] Route     : CSV
[Router] Reasoning : The question is about total revenue, which is a numerical value that can be found in the structured CSV data.
[Router] CSV hint  : Filter by 'category' = 'Electronics' and 'date' = 'December 2024', then aggregate 'total_revenue' column
[Router] Text hint : 

[Retrieving] → CSV source...

[Generating answer...]

[ANSWER]
The total revenue for the Electronics category in December 2024 was $142,864.31. This information can be found in the "Revenue by Category in December" section of the provided context.
[Saved Q1]

Question: Which region had the highest sales volume?

[Router] Route     : CSV
[Router] Reasoning : The question is about sales volume, which is a numerical metric that can be found in the structured CSV data.
[Router] CSV hint  : Use the 'units_sold' column and apply a groupby operation on the 'region' column to find the region with the highest sales

## Step 5: Questions 3–4 (Text Only)

Product detail and review questions answered from the unstructured text files.

In [9]:
# Q3-Q4
text_questions = [
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
]

for i, q in enumerate(text_questions, 3):
    answer = answer_question(q)
    print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
    print(answer)
    with open(RESULTS_FILE, "a") as f:
        f.write("Q" + str(i) + ": " + q + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")


Question: What are the key features of the Wireless Bluetooth Headphones?

[Router] Route     : TEXT
[Router] Reasoning : The question is about product features, which is best answered by the unstructured text data in the product page files.
[Router] CSV hint  : 
[Router] Text hint : Search the 'ELEC001_product_page.txt' file for keywords like 'Wireless Bluetooth Headphones' and 'features'.
[Retrieving] → Text source...

[Generating answer...]

[ANSWER]
The key features of the Wireless Bluetooth Headphones (SKU: ELEC001) are:

1. Active Noise Cancellation (ANC) with transparency mode
2. Bluetooth 5.2 for stable connectivity up to 30ft range
3. 40-hour battery life, with quick charge (10 min = 3 hours playback)
4. Premium memory foam ear cushions for all-day comfort
5. Foldable design with carrying case included
6. Built-in microphone for calls and voice assistant

As mentioned by customers, these features have provided a great user experience. For example, Sarah M. in her 5-star revie

## Step 6: Questions 5–6 (Both Sources)

Complex questions combining sales data with product review insights.

In [10]:
# Q5-Q6
both_questions = [
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

for i, q in enumerate(both_questions, 5):
    answer = answer_question(q)
    print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
    print(answer)
    with open(RESULTS_FILE, "a") as f:
        f.write("Q" + str(i) + ": " + q + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")

print("\n✅ All results saved to part2_results.txt")


Question: Which product has the best customer reviews and how well is it selling?

[Router] Route     : BOTH
[Router] Reasoning : The question requires combining sales performance from the CSV data with customer review information from the text data to determine the product with the best reviews and its sales performance.
[Router] CSV hint  : Filter by product_id and calculate total_revenue or units_sold to determine sales performance
[Router] Text hint : Search product page .txt files for keywords like 'review', 'rating', 'satisfaction' to determine customer review quality

[Retrieving] → CSV source...
[Retrieving] → Text source...

[Generating answer...]

[ANSWER]
### Product Review and Sales Performance Analysis

To determine which product has the best customer reviews and how well it is selling, we need to combine insights from both sales data and product reviews.

#### Top Products by Revenue

According to the **Total Revenue by Category** and **Top 15 Products by Revenue** data,

## (Optional) Interactive Mode

In [11]:
# Uncomment to ask your own questions
# while True:
#     q = input("Ask a question (or exit): ").strip()
#     if q.lower() in ("exit", "quit", ""):
#         break
#     print(answer_question(q))
